In [ ]:
date_start = ""   # YYYY-MM-DD or "" to use ndays
date_end   = ""   # YYYY-MM-DD exclusive, or ""
ndays         = 7      # number of observing nights
stacked_total = True   # stack cumulative state totals (False for separate lines)

In [ ]:
try:
    import rubin_nights
except ImportError:
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps --no-build-isolation > /dev/null 2>&1

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from astropy.time import Time
import astropy.units as u
from rubin_nights.observatory_status import get_dome_open_close
from scipy.stats import median_abs_deviation

try:
    get_ipython().run_line_magic('matplotlib', 'widget')
    _is_times_square = False
except Exception:
    get_ipython().run_line_magic('matplotlib', 'inline')
    _is_times_square = True

In [ ]:
from lsst.ts.xml.enums.Scheduler import ObservatoryStatus

from rubin_nights import connections

_current_location = os.getenv("EXTERNAL_INSTANCE_URL", "")
if _current_location != "":
    _tokenfile = None
    _site = None
else:
    _tokenfile = os.path.join(os.path.expanduser("~"), ".lsst/usdf_rsp")
    _site = 'usdf'
_endpoints = connections.get_clients(tokenfile=_tokenfile, site=_site)
_efd_client = _endpoints['efd']
_consdb_client = _endpoints['consdb']

DATA_NDAYS = 100
_t_end = Time.now()
_data_now = pd.Timestamp.utcnow()
_data_now_naive = _data_now.tz_localize(None)
_t_start = _t_end - DATA_NDAYS * u.day

# --- Software version check ---
_sw_ver = _efd_client.select_top_n(
    'lsst.sal.Scheduler.logevent_softwareVersions',
    ['xmlVersion', 'cscVersion'], 1)
def _ver_tuple(s):
    try:
        return tuple(int(x) for x in str(s).split('.')[:2])
    except (ValueError, AttributeError):
        return (0, 0)
_xml_ver = _ver_tuple(_sw_ver['xmlVersion'].iloc[0]) if not _sw_ver.empty else (0, 0)
_csc_ver = _ver_tuple(_sw_ver['cscVersion'].iloc[0]) if not _sw_ver.empty else (0, 0)
_new_status_logic = _xml_ver >= (27, 0) and _csc_ver >= (2, 10)

# --- Observatory status ---
df = _efd_client.select_time_series(
    'lsst.sal.Scheduler.logevent_observatoryStatus',
    ['status', 'note', 'statusLabels', 'salIndex'],
    _t_start, _t_end
)
df = df[df['salIndex'] == 1].drop(columns=['salIndex'], errors='ignore')

# --- Visits from ConsDB ---
_py_start = _t_start.to_datetime()
_py_end = _t_end.to_datetime()
_day_start = int(_py_start.strftime('%Y%m%d')) - 1
_day_end   = int(_py_end.strftime('%Y%m%d'))   + 1
visits = _consdb_client.query(f'''
    SELECT
       exp_midpt, visit1.obs_start_mjd,
       visit1.day_obs, visit1.seq_num,
       altitude, azimuth, band, img_type, visit1.exp_time,
       science_program, target_name
    FROM cdb_lsstcam.visit1
    LEFT JOIN cdb_lsstcam.visit1_quicklook
    ON visit1.visit_id = visit1_quicklook.visit_id
    WHERE visit1.day_obs >= {_day_start} AND visit1.day_obs <= {_day_end}
    ORDER BY COALESCE(exp_midpt, to_timestamp((visit1.obs_start_mjd - 40587.0) * 86400.0))
''')
visits['exp_midpt'] = pd.to_datetime(visits['exp_midpt'], utc=True, errors='coerce')
if 'obs_start_mjd' in visits.columns:
    visits['obs_start_mjd'] = pd.to_numeric(visits['obs_start_mjd'], errors='coerce')
    _missing_midpt = visits['exp_midpt'].isna() & visits['obs_start_mjd'].notna()
    if _missing_midpt.any():
        _half_s = pd.to_timedelta(visits.loc[_missing_midpt, 'exp_time'].fillna(0), unit='s') / 2
        _obs_start = pd.to_datetime(
            (visits.loc[_missing_midpt, 'obs_start_mjd'] - 40587.0) * 86400.0, unit='s', utc=True)
        visits.loc[_missing_midpt, 'exp_midpt'] = _obs_start + _half_s

# Camera shutter intervals from exposure midpoints and times
camera_shutter_intervals = []
for _, _vrow in visits.iterrows():
    if pd.notna(_vrow['exp_midpt']) and pd.notna(_vrow.get('exp_time')) and _vrow['exp_time'] > 0:
        _half = pd.Timedelta(seconds=float(_vrow['exp_time']) / 2)
        _mp = _vrow['exp_midpt']
        _t0 = (_mp - _half).tz_localize(None) if _mp.tzinfo else _mp - _half
        _t1 = (_mp + _half).tz_localize(None) if _mp.tzinfo else _mp + _half
        camera_shutter_intervals.append((_t0, _t1))


# --- Shutter open intervals ---
_dome_client = _endpoints['efd']
_dome_df = get_dome_open_close(_t_start, _t_end, _dome_client, with_sunset_sunrise=False)
shutter_open_intervals = []
for _, row in _dome_df.iterrows():
    if pd.notna(row['open_time']) and pd.notna(row['close_time']):
        ot = pd.Timestamp(row['open_time'])
        ct = pd.Timestamp(row['close_time'])
        ot = ot.tz_convert('UTC').tz_localize(None) if ot.tzinfo is not None else ot
        ct = ct.tz_convert('UTC').tz_localize(None) if ct.tzinfo is not None else ct
        shutter_open_intervals.append((ot, ct))
del _dome_df, _dome_client


# Colors: fault=vermillion, weather=gray, downtime=amber, operational=green, unknown=sky-blue
COLORS = {
    'UNKNOWN': '#56B4E9',
    'OPERATIONAL': '#009E73',
    'FAULT': '#D55E00',
    'WEATHER': '#888888',
    'DOWNTIME': '#E69F00',
    'IDLE': '#CC79A7',
}

Y_POS = {
    'UNKNOWN': 0.0,
    'OPERATIONAL': 1.0,
    'FAULT': 2.0,
    'WEATHER': 3.0,
    'DOWNTIME': 4.0,
    'IDLE': 5.0,
}

bar_states = ['UNKNOWN', 'IDLE', 'OPERATIONAL', 'DOWNTIME', 'WEATHER', 'FAULT']

# RTN-045 standard band colors and symbols
BAND_COLORS = {
    'u': '#1600ea',
    'g': '#31de1f',
    'r': '#b52626',
    'i': '#370201',
    'z': '#ba52ff',
    'y': '#61a2b3',
}
BAND_SYMBOLS = {
    'u': 'o',
    'g': '^',
    'r': 'v',
    'i': 's',
    'z': '*',
    'y': 'p',
}
BAND_ORDER = ['u', 'g', 'r', 'i', 'z', 'y']


_STATUS_BITS = [(m.value, m.name) for m in ObservatoryStatus if m.value != 0]


def decode_status(status_val):
    try:
        status_int = int(status_val)
    except (TypeError, ValueError):
        return {'UNKNOWN'}
    labels = {name for val, name in _STATUS_BITS if status_int & val}
    return labels if labels else {'UNKNOWN'}


df['labels'] = df['status'].apply(decode_status)
timestamps = df.index.tolist()
labels_list = df['labels'].tolist()
notes_list = df['note'].tolist()

# If shutter is currently open (and not daytime), add partial interval to now
_current_labels = labels_list[-1] if labels_list else set()
if 'DAYTIME' not in _current_labels:
    _shutter_now = _efd_client.select_top_n(
        'lsst.sal.MTDome.apertureShutter',
        ['positionActual0', 'positionActual1'], 1)
    if not _shutter_now.empty and _shutter_now['positionActual0'].iloc[0] > 1.0:
        _recent_shutter = _efd_client.select_time_series(
            'lsst.sal.MTDome.apertureShutter',
            ['positionActual0'], _t_end - 12*u.hour, _t_end)
        _was_closed = _recent_shutter[_recent_shutter['positionActual0'] < 1.0]
        if not _was_closed.empty:
            _open_since = _was_closed.index[-1]
        else:
            _open_since = _recent_shutter.index[0]
        _ot = pd.Timestamp(_open_since)
        _ot = _ot.tz_convert('UTC').tz_localize(None) if _ot.tzinfo else _ot
        _now_naive = _data_now_naive
        shutter_open_intervals.append((_ot, _now_naive))


# Build intervals; ignore DAYTIME
intervals = []
_downtime_active = False
for i in range(len(timestamps)):
    start = timestamps[i]
    end = timestamps[i + 1] if i + 1 < len(timestamps) else _data_now
    active = set(labels_list[i]) - {'DAYTIME'}

    if not active:
        _downtime_active = False
        continue

    if 'UNKNOWN' in active:
        _downtime_active = False  # CSC offline/standby; reset carry-forward
    else:
        # Carry DOWNTIME through WEATHER/IDLE-only gaps (status assignment bug)
        if _downtime_active and 'DOWNTIME' not in active and not (active - {'WEATHER', 'IDLE'}):
            active |= {'DOWNTIME'}
        _downtime_active = 'DOWNTIME' in active

        # WEATHER alone is an enum artifact; treat as WEATHER|UNKNOWN.
        # Safe unconditionally: new data always pairs WEATHER with IDLE.
        if active == {'WEATHER'}:
            active |= {'UNKNOWN'}

    for state in active:
        if state in COLORS:
            intervals.append((start, end, state))

# tz-naive intervals for twilight comparisons
intervals_naive = []
for start, end, state in intervals:
    s = start.tz_localize(None) if start.tzinfo is not None else start
    e = end.tz_localize(None) if end.tzinfo is not None else end
    intervals_naive.append((s, e, state))


def filter_all_during_downtime(intervals):
    dt_ivs = [(s, e) for s, e, st in intervals if st == 'DOWNTIME']
    if not dt_ivs:
        return intervals
    return [
        (s, e, st) for s, e, st in intervals
        if st == 'DOWNTIME' or not any(ds < e and de > s for ds, de in dt_ivs)
    ]

def filter_weather_during_operational(intervals):
    op_ivs = [(s, e) for s, e, st in intervals if st == 'OPERATIONAL']
    if not op_ivs:
        return intervals
    return [
        (s, e, st) for s, e, st in intervals
        if st != 'WEATHER' or not any(os < e and oe > s for os, oe in op_ivs)
    ]

def filter_unknown_with_other_states(intervals):
    other_ivs = [(s, e) for s, e, st in intervals if st != 'UNKNOWN']
    if not other_ivs:
        return intervals
    return [
        (s, e, st) for s, e, st in intervals
        if st != 'UNKNOWN' or not any(os < e and oe > s for os, oe in other_ivs)
    ]

def filter_idle_during_weather(intervals):
    wx_ivs = [(s, e) for s, e, st in intervals if st == 'WEATHER']
    if not wx_ivs:
        return intervals
    return [
        (s, e, st) for s, e, st in intervals
        if st != 'IDLE' or not any(ws < e and we > s for ws, we in wx_ivs)
    ]

def filter_weather_during_fault(intervals):
    fault_ivs = [(s, e) for s, e, st in intervals if st == 'FAULT']
    if not fault_ivs:
        return intervals
    return [
        (s, e, st) for s, e, st in intervals
        if st != 'WEATHER' or not any(fs < e and fe > s for fs, fe in fault_ivs)
    ]

def filter_idle_during_fault(intervals):
    fault_ivs = [(s, e) for s, e, st in intervals if st == 'FAULT']
    if not fault_ivs:
        return intervals
    return [
        (s, e, st) for s, e, st in intervals
        if st != 'IDLE' or not any(fs < e and fe > s for fs, fe in fault_ivs)
    ]

TOTAL_STATES = ['OPERATIONAL', 'FAULT', 'WEATHER', 'DOWNTIME', 'UNKNOWN', 'IDLE']

In [ ]:
import numpy as np
from astroplan import Observer
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import EarthLocation

# Compute -10 deg twilight for Rubin Observatory
rubin = Observer(
    location=EarthLocation.from_geodetic(-70.7494*u.deg, -30.2444*u.deg, 2663*u.m),
    name="Rubin", timezone="Chile/Continental"
)

# Generate one entry per day in the date range
start_date = timestamps[0].date()
end_date = timestamps[-1].date()
day_range = pd.date_range(start_date, end_date, freq='D')

twilight_spans = []  # (sunset_twi, sunrise_twi) in matplotlib dates — these are NIGHT spans
for day in day_range:
    t = Time(str(day.date()), scale='utc') + 12*u.hour  # noon UTC ~ morning Chile
    try:
        sunset = rubin.sun_set_time(t, which='next',
                                     horizon=-10*u.deg).to_datetime(timezone=None)
        sunrise = rubin.sun_rise_time(t + 1*u.day, which='nearest',
                                       horizon=-10*u.deg).to_datetime(timezone=None)
        twilight_spans.append((sunset, sunrise))
    except Exception:
        pass

# Configuration

In [ ]:
COMPRESS_DAYS        = True   # collapse daytime gaps to a fixed narrow width
DAY_COMPRESSED_WIDTH = 3/24   # display width of each daytime gap in days (3/24 = 3 h)
NORMALIZE_NIGHTS     = False  # make every night the same display width; note: distorts fig3 x-axis

SHOW_OFDU         = True    # include O+F+D+U sum column in the statistics table
SHOW_NIGHT_LENGTH = False   # add a night-length sub-panel below the cumulative-totals plot

DEDUP_NOTES  = True    # suppress repeated note markers when the note text is unchanged
# DEDUP_PREFIX: leading characters compared for deduplication; None = exact match.
# Set to 60 because the EFD auto-appends varying fault-component boilerplate after ~char 69,
# causing otherwise-identical human notes to look different on every status change.
DEDUP_PREFIX = 60
DEDUP_DEBUG  = False   # print each note shown or suppressed (for diagnosing dedup behaviour)

READOUT_S      = 3.4   # camera readout time in seconds (used as overhead between exposures)
MAX_SLEW_GAP_S = 120   # gaps <= this (s) between exposures counted as slew+readout overhead

FIGSIZE_TIMELINE = (18, 9) if _is_times_square else (14, 9)
FIGSIZE_CUMUL    = (18, 9) if _is_times_square else (14, 8)
FIGSIZE_TOTAL    = (18, 7) if _is_times_square else (14, 6)

import bisect
import numpy as np
import pandas as pd
import matplotlib.dates as mdates

# ── Date-range resolution ──────────────────────────────────────────────────

_ndays_end_cutoff = None
plot_start = pd.Timestamp(date_start) if date_start else None
plot_end   = pd.Timestamp(date_end)   if date_end   else None

if ndays is not None:
    _n = int(ndays)
    if plot_start is not None and plot_end is None:
        _ps = plot_start.tz_localize(None) if plot_start.tzinfo else plot_start
        _nf = [(s, r) for s, r in twilight_spans if s >= _ps]
        if len(_nf) >= _n:
            plot_start = pd.Timestamp(_nf[0][0])
            plot_end   = pd.Timestamp(_nf[_n - 1][1])
            _ndays_end_cutoff = plot_end
        else:
            plot_end = plot_start + pd.Timedelta(days=_n)
    elif plot_end is not None and plot_start is None:
        _pe = plot_end.tz_localize(None) if plot_end.tzinfo else plot_end
        _nt = [(s, r) for s, r in twilight_spans if r <= _pe]
        if len(_nt) >= _n:
            plot_start = pd.Timestamp(_nt[-_n][0])
            plot_end   = pd.Timestamp(_nt[-1][1])
            _ndays_end_cutoff = plot_end
        else:
            plot_start = plot_end - pd.Timedelta(days=_n)
    elif plot_start is None and plot_end is None:
        _now = _data_now_naive
        _past = [(s, r) for s, r in twilight_spans if r <= _now]
        _last_sr = _past[-1][1]
        _next_ss = next((s for s, r in twilight_spans if s > _last_sr), None)
        if _past:
            plot_start = pd.Timestamp(_past[max(0, len(_past) - _n)][0])
            if _next_ss is not None and _now >= _next_ss:
                plot_end = pd.Timestamp(_now)
            elif _next_ss is not None:
                plot_end = pd.Timestamp(_next_ss)
            else:
                plot_end = _now
            _ndays_end_cutoff = plot_end
        else:
            plot_end   = timestamps[-1]
            plot_start = plot_end - pd.Timedelta(days=_n)

if plot_start is None:
    plot_start = timestamps[0]
if plot_end is None:
    plot_end = timestamps[-1]
if plot_start.tzinfo is None and timestamps[0].tzinfo is not None:
    plot_start = plot_start.tz_localize(timestamps[0].tzinfo)
if plot_end.tzinfo is None and timestamps[0].tzinfo is not None:
    plot_end = plot_end.tz_localize(timestamps[0].tzinfo)

# ── Plot x-limits and twilight boundaries ─────────────────────────────────

xlim0 = mdates.date2num(plot_start - pd.Timedelta(hours=2))
xlim1 = mdates.date2num(plot_end) + 2/24

all_boundaries    = []
plot_twilight_spans = []
for _ss, _sr in twilight_spans:
    _sn, _rn = mdates.date2num(_ss), mdates.date2num(_sr)
    if _rn >= xlim0 and _sn <= xlim1:
        if _ndays_end_cutoff is not None and _ss >= _ndays_end_cutoff:
            continue
        all_boundaries.append((_sn, _rn))
        plot_twilight_spans.append((_ss, _sr))

# ── Piecewise-linear compression (cx / uncx) ──────────────────────────────

_segments = []
_prev_end = xlim0
for _ns, _nr in all_boundaries:
    if _ns > _prev_end:
        _segments.append((_prev_end, _ns, False))   # daytime
    _segments.append((_ns, _nr, True))               # nighttime
    _prev_end = _nr
if _prev_end < xlim1:
    _segments.append((_prev_end, xlim1, False))

if COMPRESS_DAYS:
    _real_bp = [xlim0]
    _comp_bp = [0.0]
    _cpos    = 0.0
    if NORMALIZE_NIGHTS:
        _nws = [re - rs for rs, re, inight in _segments if inight]
        _night_display_w = np.mean(_nws) if _nws else 0.5
    for rs, re, inight in _segments:
        if inight:
            _cpos += _night_display_w if NORMALIZE_NIGHTS else (re - rs)
        else:
            _cpos += DAY_COMPRESSED_WIDTH
        _real_bp.append(re)
        _comp_bp.append(_cpos)
    _real_bp = np.array(_real_bp)
    _comp_bp = np.array(_comp_bp)

    def cx(x):
        return np.interp(x, _real_bp, _comp_bp)

    def uncx(x):
        return np.interp(x, _comp_bp, _real_bp)

    cxlim0, cxlim1 = float(cx(xlim0)), float(cx(xlim1))
else:
    def cx(x):   return x
    def uncx(x): return x
    cxlim0, cxlim1 = xlim0, xlim1

# ── Tick / interval helpers ────────────────────────────────────────────────

tick_start = plot_start.date() if hasattr(plot_start, 'date') else plot_start
tick_end   = plot_end.date()   if hasattr(plot_end,   'date') else plot_end

_interval_starts = [s for s, e, state in intervals_naive]


def shade_daytime(ax):
    _pe = cxlim0
    for _ns, _nr in all_boundaries:
        _cn = float(cx(_ns))
        _cr = float(cx(_nr))
        if _cn > _pe:
            ax.axvspan(_pe, _cn, color='#fff9c4', alpha=0.6, zorder=0)
        _pe = _cr
    if _pe < cxlim1:
        ax.axvspan(_pe, cxlim1, color='#fff9c4', alpha=0.6, zorder=0)


def setup_xaxis(ax, show_labels=False):
    if COMPRESS_DAYS:
        _span_h = (plot_end - plot_start).total_seconds() / 3600
        if _span_h < 48:
            _dt0 = plot_start.tz_localize(None) if plot_start.tzinfo else plot_start
            _dt1 = plot_end.tz_localize(None)   if plot_end.tzinfo   else plot_end
            if _span_h <= 6:
                _freq, _fmt = '30min', '%H:%M'
            elif _span_h <= 24:
                _freq, _fmt = '1h',    '%H:%M'
            else:
                _freq, _fmt = '3h',    '%m-%d %H:%M'
            _tds = pd.date_range(_dt0.floor('h') - pd.Timedelta(hours=1),
                                  _dt1.ceil('h')  + pd.Timedelta(hours=1), freq=_freq)
            ax.set_xticks([float(cx(mdates.date2num(d))) for d in _tds])
            ax.set_xticks([], minor=True)
            ax.set_xticklabels([d.strftime(_fmt) for d in _tds] if show_labels else [])
        else:
            _td2 = pd.date_range(tick_start, tick_end, freq='2D')
            ax.set_xticks([float(cx(mdates.date2num(d + pd.Timedelta(hours=12)))) for d in _td2])
            ax.set_xticklabels([d.strftime('%m-%d') for d in _td2] if show_labels else [])
            _td1 = pd.date_range(tick_start, tick_end, freq='D')
            ax.set_xticks([float(cx(mdates.date2num(d + pd.Timedelta(hours=12)))) for d in _td1],
                          minor=True)
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
        ax.xaxis.set_minor_locator(mdates.DayLocator())
        if not show_labels:
            ax.set_xticklabels([])
    ax.set_xlim(cxlim0, cxlim1)
    ax.fmt_xdata = lambda x: mdates.num2date(uncx(x)).strftime('%Y-%m-%d %H:%M:%S')
    ax.grid(axis='x', alpha=0.3)


def iter_status_events():
    """Yield (ts_naive, ts_cx, active_labels, text, has_note) for each status change.

    With DEDUP_NOTES=True a diamond marker is shown only on the first occurrence
    of each note text. The same text never shows twice, even across empty-note events.
    A genuinely new note text always shows a diamond.
    """
    _last_shown_note = None
    for i in range(len(timestamps)):
        raw_labels = labels_list[i]
        if 'UNKNOWN' in raw_labels:
            active = {'UNKNOWN'}
        else:
            active = raw_labels - {'DAYTIME'}
            if not active:
                continue

        ts       = timestamps[i]
        ts_naive = ts.tz_localize(None) if ts.tzinfo is not None else ts
        ts_cx    = float(cx(mdates.date2num(ts)))

        raw_note = notes_list[i] if pd.notna(notes_list[i]) else ''
        note = ' '.join(str(raw_note).split())  # collapse all whitespace variants
        if DEDUP_NOTES:
            _dp = globals().get('DEDUP_PREFIX', 60)
            if not note:
                has_note = False
            elif _last_shown_note is None:
                has_note = True
            elif _dp is None:
                has_note = note != _last_shown_note
            else:
                _n = min(len(note), len(_last_shown_note), _dp)
                has_note = note[:_n] != _last_shown_note[:_n]
            if globals().get('DEDUP_DEBUG', False):
                if has_note:
                    print(f"  NOTE shown  @ {ts_naive}: {note[:80]!r}")
                elif bool(note):
                    print(f"  NOTE suppressed @ {ts_naive}: {note[:80]!r}")
            if has_note:
                _last_shown_note = note
        else:
            has_note = bool(note)

        label_str = ', '.join(sorted(active))
        text = f"{label_str}\n{note}" if note else label_str

        yield ts_naive, ts_cx, active, text, has_note


def make_hover_annot(ax):
    """Return (annot, update_fn(visible, xy, text))."""
    annot = ax.annotate('', xy=(0, 0), xytext=(10, 15),
                        textcoords='offset points',
                        bbox=dict(boxstyle='round,pad=0.3', fc='white',
                                  ec='gray', alpha=0.95),
                        fontsize=7, visible=False, zorder=10)

    def update(visible, xy=None, text=None):
        if visible:
            annot.xy = xy
            lines = text.split('\n')
            annot.set_text('\n'.join(
                l[:100] + '...' if len(l) > 100 else l for l in lines))
            xlim = ax.get_xlim()
            frac = ((xy[0] - xlim[0]) / (xlim[1] - xlim[0])
                    if xlim[1] != xlim[0] else 0)
            if frac > 0.75:
                annot.set_position((-10, 15)); annot.set_ha('right')
            else:
                annot.set_position(( 10, 15)); annot.set_ha('left')
        annot.set_visible(visible)

    return annot, update


def _format_coord_with_states(x, y):
    dt     = mdates.num2date(uncx(x)).replace(tzinfo=None)
    ts_str = dt.strftime('%Y-%m-%d %H:%M:%S')
    active = set()
    idx    = bisect.bisect_right(_interval_starts, dt)
    for i in range(idx - 1, -1, -1):
        s, e, state = intervals_naive[i]
        if e <= dt:
            break
        if s <= dt < e:
            active.add(state)
    return f"{ts_str}  [{', '.join(sorted(active))}]" if active else ts_str


title_start = plot_start.strftime('%Y-%m-%d')
title_end   = plot_end.strftime('%Y-%m-%d')

from IPython.display import clear_output
clear_output()


# Make timeline plot

In [ ]:

# Close previous figure if re-running
try:
    plt.close(fig)
except NameError:
    pass

# Order: Alt&Az (top), Band (middle), Status (bottom)
fig, (ax_altaz, ax_band, ax_status) = plt.subplots(
    3, 1, figsize=FIGSIZE_TIMELINE, sharex=True,
    gridspec_kw={'height_ratios': [0.8, 1.2, 5], 'hspace': 0})
fig.subplots_adjust(bottom=0.12)

# --- Panel 1 (top): Altitude and Azimuth ---
shade_daytime(ax_altaz)

visit_x_real = mdates.date2num(visits['exp_midpt'].values.astype('datetime64[ns]'))
visit_x = cx(visit_x_real)

ax_altaz.scatter(visit_x, visits['altitude'].values, c='red', s=3, alpha=0.5, zorder=3)
ax_az = ax_altaz.twinx()
ax_az.scatter(visit_x, visits['azimuth'].values, c='blue', s=3, alpha=0.3, zorder=3)
ax_az.fmt_xdata = lambda x: mdates.num2date(uncx(x)).strftime('%Y-%m-%d %H:%M:%S')

def _format_coord_altaz(x, y):
    real_x = uncx(x)
    dt = mdates.num2date(real_x).replace(tzinfo=None)
    ts_str = dt.strftime('%Y-%m-%d %H:%M:%S')
    # Find nearest visit
    if len(visit_x) > 0:
        dists = np.abs(visit_x - x)
        nearest = np.argmin(dists)
        alt = visits['altitude'].values[nearest]
        az = visits['azimuth'].values[nearest]
        return f"{ts_str}  [alt={alt:.1f}, az={az:.1f}]"
    return ts_str

ax_az.format_coord = _format_coord_altaz

ax_altaz.set_ylabel('Alt', color='red', fontsize=7)
ax_altaz.set_ylim(-5, 95)
ax_altaz.tick_params(axis='y', labelcolor='red', labelsize=6)
ax_az.set_ylabel('Az', color='blue', fontsize=7)
ax_az.set_ylim(-20, 380)
ax_az.tick_params(axis='y', labelcolor='blue', labelsize=6)

from matplotlib.lines import Line2D
leg_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=4, label='Alt'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=4, label='Az'),
]
ax_altaz.legend(handles=leg_handles, loc='upper right', ncol=2, fontsize=7)
ax_altaz.set_title(f'Observatory Status Timeline ({title_start} to {title_end} UTC)', fontsize=10)
setup_xaxis(ax_altaz, show_labels=False)
ax_altaz.grid(False, axis="x")

# --- Panel 2 (middle): Band ---
shade_daytime(ax_band)

is_science = (visits['img_type'] == 'science').values
bands = visits['band'].fillna('').values
img_types = visits['img_type'].values
sci_progs = visits['science_program'].fillna('').values
seq_nums = visits['seq_num'].apply(lambda x: str(int(x)) if pd.notna(x) else '').values
target_names = visits['target_name'].fillna('').values

# y positions: science=1, other=0
band_scatter_y = np.where(is_science, 0.7, 0.0)

# Build tooltip texts
band_scatter_texts = []
for band, itype, prog, seq, tgt in zip(bands, img_types, sci_progs, seq_nums, target_names):
    parts = [itype]
    if band:
        parts.append(band)
    if prog:
        parts.append(prog)
    if seq != '':
        parts.append(f'seq {seq}')
    if tgt:
        parts.append(tgt)
    band_scatter_texts.append(': '.join(parts))

# Plot each band separately with its RTN-045 color and symbol
band_scatters = []
for b in BAND_ORDER:
    mask = bands == b
    if not np.any(mask):
        continue
    sc = ax_band.scatter(
        visit_x[mask], band_scatter_y[mask],
        c=BAND_COLORS[b], marker=BAND_SYMBOLS[b],
        s=np.where(is_science[mask], 20, 8),
        edgecolors='none', alpha=0.7, zorder=3, label=b)
    band_scatters.append((sc, np.where(mask)[0]))

# Handle visits with unknown/missing band
other_mask = ~np.isin(bands, BAND_ORDER)
if np.any(other_mask):
    sc = ax_band.scatter(
        visit_x[other_mask], band_scatter_y[other_mask],
        c='#999999', marker='x', s=4,
        alpha=0.5, zorder=3, label='?')
    band_scatters.append((sc, np.where(other_mask)[0]))

ax_band.set_ylim(-0.4, 2.2)
ax_band.set_autoscaley_on(False)
ax_band.set_yticks([0, 0.7, 1.4])
ax_band.set_yticklabels(['other', 'science', 'dome shutter'], fontsize=7)

band_legend = [
    Line2D([0], [0], marker=BAND_SYMBOLS[b], color='w',
           markerfacecolor=BAND_COLORS[b], markersize=5, label=b)
    for b in BAND_ORDER
]
band_legend.append(Patch(facecolor='#CC79A7', alpha=0.5, label='dome open'))
ax_band.legend(handles=band_legend, loc='upper right', ncol=7, fontsize=6)
setup_xaxis(ax_band, show_labels=False)
ax_band.grid(False, axis="x")
# Show shutter-open intervals as green bar at y=1.5
for s_start, s_end in shutter_open_intervals:
    x0 = float(cx(mdates.date2num(s_start)))
    x1 = float(cx(mdates.date2num(s_end)))
    ax_band.barh(1.4, x1 - x0, left=x0, height=0.3,
           color='#CC79A7', alpha=0.5, edgecolor='none', zorder=2)
ax_band.format_coord = _format_coord_with_states

# --- Panel 3 (bottom): Observatory Status ---
y_min, y_max = -0.5, 4.5
shade_daytime(ax_status)

for state in bar_states:
    for start, end, s in intervals:
        if s != state:
            continue
        x0 = float(cx(mdates.date2num(start)))
        x1 = float(cx(mdates.date2num(end)))
        width = x1 - x0
        if state == 'DOWNTIME':
            ax_status.barh((y_min + y_max)/2, width, left=x0, height=y_max - y_min,
                           color='none', edgecolor=COLORS[state], linewidth=0,
                           hatch='///', alpha=0.4, zorder=2)
        else:
            ax_status.barh((y_min + y_max)/2, width, left=x0, height=y_max - y_min,
                           color=COLORS[state], edgecolor='none', alpha=0.3, zorder=2)

priority = ['FAULT', 'WEATHER', 'DOWNTIME', 'OPERATIONAL', 'UNKNOWN']

# Split into note/no-note for different markers (diamond=note, circle=normal)
note_x, note_y, note_c, note_texts = [], [], [], []
norm_x, norm_y, norm_c, norm_texts = [], [], [], []
scatter_x, scatter_y, scatter_texts = [], [], []  # combined for hover lookup
for ts_naive, ts_cx, active, text, has_note in iter_status_events():
    for state in active:
        if state in Y_POS:
            c = COLORS.get(state, '#999')
            scatter_x.append(ts_cx)
            scatter_y.append(Y_POS[state])
            scatter_texts.append(text)
            if has_note:
                _ns = next((s for s in priority if s in active and s in Y_POS),
                           next((s for s in active if s in Y_POS), None))
                if state == _ns:
                    note_x.append(ts_cx)
                    note_y.append(Y_POS[state])
                    note_c.append(c)
                    note_texts.append(text)
                else:
                    norm_x.append(ts_cx)
                    norm_y.append(Y_POS[state])
                    norm_c.append(c)
                    norm_texts.append(text)
            else:
                norm_x.append(ts_cx)
                norm_y.append(Y_POS[state])
                norm_c.append(c)
                norm_texts.append(text)

sc1_norm = ax_status.scatter(norm_x, norm_y, c=norm_c, s=40,
                              marker='o', zorder=5, edgecolors='black', linewidths=0.3)
sc1_note = ax_status.scatter(note_x, note_y, c=note_c, s=50,
                              marker='D', zorder=6, edgecolors='black', linewidths=0.5)

ax_status.set_ylim(y_min, y_max)
ax_status.set_yticks([Y_POS[s] for s in ['UNKNOWN', 'OPERATIONAL', 'FAULT', 'WEATHER', 'DOWNTIME']])
ax_status.set_yticklabels(['UNKNOWN', 'OPERATIONAL', 'FAULT', 'WEATHER', 'DOWNTIME'])

legend_items = [
    Patch(facecolor='#fff9c4', label='DAYTIME'),
    Patch(facecolor=COLORS['OPERATIONAL'], alpha=0.3, label='OPERATIONAL'),
    Patch(facecolor=COLORS['FAULT'], alpha=0.3, label='FAULT'),
    Patch(facecolor=COLORS['WEATHER'], alpha=0.3, label='WEATHER'),
    Patch(facecolor='white', edgecolor=COLORS['DOWNTIME'], hatch='///', label='DOWNTIME'),
    Patch(facecolor=COLORS['UNKNOWN'], alpha=0.3, label='UNKNOWN'),
]
ax_status.legend(handles=legend_items, loc='upper right', ncol=6, fontsize=8)
setup_xaxis(ax_status, show_labels=True)
ax_status.set_xlabel('Date (UTC)')
plt.setp(ax_status.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax_status.format_coord = _format_coord_with_states

# Mouseover annotations
_, update_status = make_hover_annot(ax_status)
_, update_band = make_hover_annot(ax_band)
_hover_state = [None, None]  # [status_key, band_key]


def on_hover(event):
    changed = False
    if event.inaxes == ax_status:
        found = False
        for sc, texts, xs, ys in [(sc1_note, note_texts, note_x, note_y),
                                   (sc1_norm, norm_texts, norm_x, norm_y)]:
            cont, ind = sc.contains(event)
            if cont:
                i = ind['ind'][0]
                key = (id(sc), i)
                if key != _hover_state[0]:
                    update_status(True, (xs[i], ys[i]), texts[i])
                    _hover_state[0] = key
                    changed = True
                found = True
                break
        if not found and _hover_state[0] is not None:
            update_status(False)
            _hover_state[0] = None
            changed = True
    elif _hover_state[0] is not None:
        update_status(False)
        _hover_state[0] = None
        changed = True

    if event.inaxes == ax_band:
        found = False
        for sc, indices in band_scatters:
            cont, ind = sc.contains(event)
            if cont:
                orig_idx = indices[ind['ind'][0]]
                key = ('band', orig_idx)
                if key != _hover_state[1]:
                    update_band(True,
                                (float(visit_x[orig_idx]), float(band_scatter_y[orig_idx])),
                                band_scatter_texts[orig_idx])
                    _hover_state[1] = key
                    changed = True
                found = True
                break
        if not found and _hover_state[1] is not None:
            update_band(False)
            _hover_state[1] = None
            changed = True
    elif _hover_state[1] is not None:
        update_band(False)
        _hover_state[1] = None
        changed = True

    if changed:
        fig.canvas.draw_idle()

fig.canvas.mpl_connect('motion_notify_event', on_hover)

plt.show()

# Make plots which are cumulative over a number of days

In [ ]:
# Cumulative time in each state per observing night
# An "observing night" runs from one -10deg sunset to the next -10deg sunrise

STACKED_CUMUL = False   # True = stacked filled areas, False = independent lines
SHADE_CURVES = True     # Shade under individual curves (applies when STACKED_CUMUL is False)
CUMUL_STATES = ['OPERATIONAL', 'FAULT', 'WEATHER', 'DOWNTIME', 'UNKNOWN', 'IDLE']
# States to stack; DOWNTIME drawn as line; WEATHER drawn as independent filled area
LINE_ONLY_STATES = {} # {'DOWNTIME'}
FILL_ONLY_STATES = {'WEATHER'}  # filled but not stacked
STACK_STATES = [s for s in CUMUL_STATES if s not in LINE_ONLY_STATES and s not in FILL_ONLY_STATES]

CUMUL_LINESTYLE = {
    'OPERATIONAL': '-',
    'FAULT': '-',
    'WEATHER': '-',
    'DOWNTIME': '--',
    'UNKNOWN': '-',
    'IDLE': '-',
}

CUMUL_COLORS = dict(COLORS)

# Close previous figure if re-running
try:
    plt.close(fig2)
except NameError:
    pass

fig2, ax_cumul = plt.subplots(figsize=FIGSIZE_CUMUL)

shade_daytime(ax_cumul)

# Store cumulative curve data per night for scatter point lookup
night_curves = []

for sunset, sunrise in plot_twilight_spans:
    sunset_pd = pd.Timestamp(sunset)
    sunrise_pd = min(pd.Timestamp(sunrise), _data_now_naive)
    night_hours = (sunrise_pd - sunset_pd).total_seconds() / 3600.0

    night_intervals = []
    for start, end, state in intervals_naive:
        if state not in CUMUL_STATES:
            continue
        s = max(start, sunset_pd)
        e = min(end, sunrise_pd)
        if s < e:
            night_intervals.append((s, e, state))

    night_intervals = filter_all_during_downtime(night_intervals)
    night_intervals = filter_weather_during_operational(night_intervals)
    night_intervals = filter_unknown_with_other_states(night_intervals)
    night_intervals = filter_idle_during_weather(night_intervals)
    night_intervals = filter_weather_during_fault(night_intervals)
    night_intervals = filter_idle_during_fault(night_intervals)

    if not night_intervals:
        continue

    night_intervals.sort(key=lambda x: x[0])

    cumul = {st: 0.0 for st in CUMUL_STATES}
    line_data = {st: [] for st in CUMUL_STATES}

    sunset_cx = float(cx(mdates.date2num(sunset_pd)))
    for st in CUMUL_STATES:
        line_data[st].append((sunset_cx, 0.0))

    for s, e, state in night_intervals:
        dt_hours = (e - s).total_seconds() / 3600.0
        t0 = float(cx(mdates.date2num(s)))
        t1 = float(cx(mdates.date2num(e)))
        old_val = cumul[state]
        new_val = old_val + dt_hours
        line_data[state].append((t0, old_val))
        line_data[state].append((t1, new_val))
        cumul[state] = new_val

    sunrise_cx = float(cx(mdates.date2num(sunrise_pd)))
    for st in CUMUL_STATES:
        if cumul[st] > 0:
            line_data[st].append((sunrise_cx, cumul[st]))

    # Build per-state curves (xs, ys arrays)
    curves = {}
    for st in CUMUL_STATES:
        if cumul[st] > 0:
            pts = line_data[st]
            xs = np.array([p[0] for p in pts])
            ys = np.array([p[1] for p in pts])
            curves[st] = (xs, ys)

    if STACKED_CUMUL:
        # Merge all x-breakpoints across stackable states for this night
        all_xs = set([sunset_cx, sunrise_cx])
        for st in STACK_STATES:
            if st in curves:
                all_xs.update(curves[st][0].tolist())
        all_xs = np.array(sorted(all_xs))

        # Interpolate each stackable state at all breakpoints
        interp_vals = {}
        for st in STACK_STATES:
            if st in curves:
                interp_vals[st] = np.interp(all_xs, curves[st][0], curves[st][1])
            else:
                interp_vals[st] = np.zeros_like(all_xs)

        # Draw stacked fill_between (bottom to top per STACK_STATES order)
        bottom = np.zeros_like(all_xs)
        stacked_tops = {}
        for st in STACK_STATES:
            top = bottom + interp_vals[st]
            stacked_tops[st] = (all_xs, top.copy())
            if np.any(interp_vals[st] > 0):
                ax_cumul.fill_between(all_xs, bottom, top, color=CUMUL_COLORS[st],
                                       alpha=0.7, step=None, zorder=2,
                                       linewidth=0.5, edgecolor=CUMUL_COLORS[st])
            bottom = top

        # Show stacked total line when there's significant unaccounted time
        stacked_total = bottom[-1]
        if night_hours - stacked_total > 0.1:
            ax_cumul.plot([sunset_cx, sunrise_cx], [stacked_total, stacked_total],
                          color='gray', linewidth=1.0, linestyle='-', alpha=0.7, zorder=4)

        # Draw fill-only states as independent filled areas (not stacked)
        for st in FILL_ONLY_STATES:
            if st in curves:
                xs, ys = curves[st]
                ax_cumul.fill_between(xs, 0, ys, color=CUMUL_COLORS[st],
                                       alpha=0.5, zorder=2,
                                       linewidth=0.5, edgecolor=CUMUL_COLORS[st])

        # Draw line-only states as lines
        for st in LINE_ONLY_STATES:
            if st in curves:
                xs, ys = curves[st]
                ax_cumul.plot(xs, ys, color=CUMUL_COLORS[st], linewidth=1.5,
                               linestyle=CUMUL_LINESTYLE[st], alpha=0.9, zorder=3)

        night_curves.append((sunset_pd, sunrise_pd, sunset_cx, sunrise_cx,
                             curves, stacked_tops))
    else:
        for st in CUMUL_STATES:
            if st in curves:
                xs, ys = curves[st]
                ax_cumul.plot(xs, ys, color=CUMUL_COLORS[st], linewidth=1.5,
                               linestyle=CUMUL_LINESTYLE[st], alpha=0.9, zorder=3)
                if SHADE_CURVES:
                    if st in LINE_ONLY_STATES:
                        ax_cumul.fill_between(xs, 0, ys, facecolor='none',
                                               edgecolor=CUMUL_COLORS[st],
                                               hatch='///', alpha=0.3, zorder=1)
                    else:
                        ax_cumul.fill_between(xs, 0, ys, color=CUMUL_COLORS[st],
                                               alpha=0.2, zorder=1)

        night_curves.append((sunset_pd, sunrise_pd, sunset_cx, sunrise_cx,
                             curves, None))

    # Night-length line
    ax_cumul.plot([sunset_cx, sunrise_cx], [night_hours, night_hours],
                  color='black', linewidth=1.5, solid_capstyle='butt', zorder=4)

    # Top-down band for DOWNTIME total
    xs_band = [sunset_cx, sunrise_cx]
    if cumul.get('DOWNTIME', 0) > 0:
        ax_cumul.fill_between(xs_band, night_hours - cumul['DOWNTIME'], night_hours,
                               facecolor='none', edgecolor=CUMUL_COLORS['DOWNTIME'],
                               hatch='///', alpha=0.4, zorder=3.5)

    ax_cumul.axvline(sunset_cx, color='orange', linewidth=0.3, alpha=0.5, zorder=1)
    ax_cumul.axvline(sunrise_cx, color='orange', linewidth=0.3, alpha=0.5, zorder=1)

    # Cumulative shutter-open time for this night
    shutter_night = []
    for s_start, s_end in shutter_open_intervals:
        s = max(s_start, sunset_pd)
        e = min(s_end, sunrise_pd)
        if s < e:
            shutter_night.append((s, e))
    if shutter_night:
        shutter_night.sort()
        sh_cumul = 0.0
        sh_xs = [sunset_cx]
        sh_ys = [0.0]
        for s, e in shutter_night:
            t0 = float(cx(mdates.date2num(s)))
            t1 = float(cx(mdates.date2num(e)))
            dt_h = (e - s).total_seconds() / 3600.0
            sh_xs.extend([t0, t1])
            sh_ys.extend([sh_cumul, sh_cumul + dt_h])
            sh_cumul += dt_h
        sh_xs.append(sunrise_cx)
        sh_ys.append(sh_cumul)
        ax_cumul.plot(sh_xs, sh_ys, color='#CC79A7', linewidth=1.5,
                      linestyle='-', alpha=0.9, zorder=3)

# Overlay status scatter points ON the cumulative curves
note_x2, note_y2, note_c2, note_texts2 = [], [], [], []
norm_x2, norm_y2, norm_c2, norm_texts2 = [], [], [], []
scatter_x2, scatter_y2, scatter_texts2 = [], [], []
for ts_naive, ts_cx, active, text, has_note in iter_status_events():
    for entry in night_curves:
        sunset_pd, sunrise_pd, sunset_cx, sunrise_cx, curves, stacked_tops = entry
        if sunset_pd <= ts_naive <= sunrise_pd:
            for state in active:
                if state in curves:
                    if STACKED_CUMUL and stacked_tops is not None and state in stacked_tops:
                        xs, ys = stacked_tops[state]
                        y_val = float(np.interp(ts_cx, xs, ys))
                    else:
                        xs, ys = curves[state]
                        y_val = float(np.interp(ts_cx, xs, ys))
                    c = CUMUL_COLORS.get(state, '#999')
                    scatter_x2.append(ts_cx)
                    scatter_y2.append(y_val)
                    scatter_texts2.append(text)
                    if has_note:
                        _ns2 = next((s for s in priority if s in active and s in curves),
                                    next((s for s in active if s in curves), None))
                        if state == _ns2:
                            note_x2.append(ts_cx)
                            note_y2.append(y_val)
                            note_c2.append(c)
                            note_texts2.append(text)
                        else:
                            norm_x2.append(ts_cx)
                            norm_y2.append(y_val)
                            norm_c2.append(c)
                            norm_texts2.append(text)
                    else:
                        norm_x2.append(ts_cx)
                        norm_y2.append(y_val)
                        norm_c2.append(c)
                        norm_texts2.append(text)
            break

sc2_norm = ax_cumul.scatter(norm_x2, norm_y2, c=norm_c2, s=40,
                             marker='o', zorder=5, edgecolors='black', linewidths=0.2)
sc2_note = ax_cumul.scatter(note_x2, note_y2, c=note_c2, s=50,
                             marker='D', zorder=6, edgecolors='black', linewidths=0.4)

max_night_hours = max((sunrise - sunset).total_seconds() / 3600
                       for sunset, sunrise in plot_twilight_spans)
ax_cumul.set_ylim(0, max_night_hours + 1.5)
ax_cumul.set_ylabel('Cumulative hours', fontsize=8)
_cumul_title = 'Cumulative Time in State per Night'
if STACKED_CUMUL:
    _cumul_title += ' (stacked)'
ax_cumul.set_title(_cumul_title, fontsize=10, loc='left')

setup_xaxis(ax_cumul, show_labels=True)
ax_cumul.set_xlabel('Date (UTC)')
plt.setp(ax_cumul.xaxis.get_majorticklabels(), rotation=45, ha='right')
fig2.subplots_adjust(bottom=0.15)

# Mouseover annotation
_, update_cumul = make_hover_annot(ax_cumul)
_hover_state2 = [None]


def on_hover2(event):
    changed = False
    if event.inaxes == ax_cumul:
        found = False
        for sc, texts, xs, ys in [(sc2_note, note_texts2, note_x2, note_y2),
                                   (sc2_norm, norm_texts2, norm_x2, norm_y2)]:
            cont, ind = sc.contains(event)
            if cont:
                i = ind['ind'][0]
                key = (id(sc), i)
                if key != _hover_state2[0]:
                    update_cumul(True, (xs[i], ys[i]), texts[i])
                    _hover_state2[0] = key
                    changed = True
                found = True
                break
        if not found and _hover_state2[0] is not None:
            update_cumul(False)
            _hover_state2[0] = None
            changed = True
    elif _hover_state2[0] is not None:
        update_cumul(False)
        _hover_state2[0] = None
        changed = True
    if changed:
        fig2.canvas.draw_idle()


fig2.canvas.mpl_connect('motion_notify_event', on_hover2)

from matplotlib.lines import Line2D
if STACKED_CUMUL:
    cumul_legend = [
        Patch(facecolor=CUMUL_COLORS[st], alpha=0.7, label=st) for st in STACK_STATES
    ] + [
        Patch(facecolor=CUMUL_COLORS[st], alpha=0.5, label=st) for st in FILL_ONLY_STATES
    ] + [
        Line2D([0], [0], color=CUMUL_COLORS[st], linewidth=2,
               linestyle=CUMUL_LINESTYLE[st], label=st) for st in LINE_ONLY_STATES
    ] + [Line2D([0], [0], color='black', linewidth=1.5, label='Night length')]
else:
    cumul_legend = [
        Line2D([0], [0], color=CUMUL_COLORS[st], linewidth=2,
               linestyle=CUMUL_LINESTYLE[st], label=st) for st in CUMUL_STATES
    ] + [Line2D([0], [0], color='black', linewidth=1.5, label='Night length')]
# Add top-band legend entry
cumul_legend += [
    Patch(facecolor='none', edgecolor=CUMUL_COLORS['DOWNTIME'], hatch='///',
          label='DOWNTIME (total)'),
]
cumul_legend += [
    Line2D([0], [0], color='#CC79A7', linewidth=1.5, label='Dome open'),
]
ax_cumul.legend(handles=cumul_legend, loc='upper right', ncol=4, fontsize=7)

plt.show()

In [ ]:
# Sync x-axis zoom/pan between fig (timeline) and fig2 (cumulative)
_xlim_syncing = False

# Persistent "UTC" text at lower-right of each bottom axis
_utc_label_status = ax_status.text(1.0, -0.01, 'UTC', transform=ax_status.transAxes,
                                    fontsize=8, ha='right', va='top', color='black')
_utc_label_cumul = ax_cumul.text(1.0, -0.01, 'UTC', transform=ax_cumul.transAxes,
                                  fontsize=8, ha='right', va='top', color='black')

def _update_titles(xlim):
    """Update both figure titles to reflect the visible date range."""
    d0 = mdates.num2date(uncx(xlim[0])).strftime('%Y-%m-%d')
    d1 = mdates.num2date(uncx(xlim[1])).strftime('%Y-%m-%d')
    ax_altaz.set_title(f'Observatory Status Timeline ({d0} to {d1} UTC)', fontsize=10)
    _ct = 'Cumulative Time in State per Night'
    if STACKED_CUMUL:
        _ct += ' (stacked)'
    ax_cumul.set_title(f'{_ct} ({d0} to {d1} UTC)', fontsize=10, loc='left')
    try:
        _stacked_suffix = ' (stacked)' if STACKED_TOTAL else ''
        ax_total.set_title(f'Cumulative State Totals{_stacked_suffix} ({d0} to {d1} UTC)', fontsize=10)
    except NameError:
        pass

def _update_ticks(xlim):
    """Recompute x-axis ticks so at least 5 are visible in the current view."""
    r0 = uncx(xlim[0])
    r1 = uncx(xlim[1])
    dt0 = mdates.num2date(r0).replace(tzinfo=None)
    dt1 = mdates.num2date(r1).replace(tzinfo=None)
    span_sec = (dt1 - dt0).total_seconds()
    if span_sec <= 0:
        return

    # Coarsest to finest; pick the first (coarsest) that gives >= 5 ticks
    freq_options = [
        ('D', 14), ('D', 7), ('D', 5), ('D', 2), ('D', 1),
        ('h', 12), ('h', 6), ('h', 3), ('h', 2), ('h', 1),
        ('min', 30), ('min', 15), ('min', 10), ('min', 5), ('min', 2), ('min', 1),
    ]
    chosen_freq = ('D', 2)  # fallback
    fmt = '%m-%d'
    for unit, val in freq_options:
        if unit == 'min':
            step_sec = val * 60
        elif unit == 'h':
            step_sec = val * 3600
        else:
            step_sec = val * 86400
        n_ticks = span_sec / step_sec
        if n_ticks >= 5:
            chosen_freq = (unit, val)
            if unit in ('min', 'h'):
                fmt = '%m-%d %H:%M'
            break

    unit, val = chosen_freq
    if unit == 'min':
        freq_str = f'{val}min'
    elif unit == 'h':
        freq_str = f'{val}h'
    else:
        freq_str = f'{val}D'

    tick_dts = pd.date_range(dt0.replace(minute=0, second=0, microsecond=0),
                              dt1 + pd.Timedelta(hours=1), freq=freq_str)
    tick_dts = tick_dts[(tick_dts >= dt0) & (tick_dts <= dt1)]

    if unit == 'D':
        tick_pos = [float(cx(mdates.date2num(d + pd.Timedelta(hours=12)))) for d in tick_dts]
    else:
        tick_pos = [float(cx(mdates.date2num(d))) for d in tick_dts]
    tick_lab = [d.strftime(fmt) for d in tick_dts]

    _tick_axes = [ax_status, ax_cumul]
    try:
        _tick_axes.append(ax_total)
    except NameError:
        pass
    for ax in _tick_axes:
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_lab, rotation=45, ha='right')
        ax.set_xticks([], minor=True)

    # Keep labels only on bottom panel; hide on upper panels without affecting shared state
    ax_altaz.tick_params(axis='x', labelbottom=False)
    ax_band.tick_params(axis='x', labelbottom=False)
    # Disable x-grid on upper panels so tick positions don't draw vertical lines
    ax_altaz.grid(False, axis='x')
    ax_band.grid(False, axis='x')

def _sync_from_timeline(ax):
    global _xlim_syncing
    if _xlim_syncing:
        return
    _xlim_syncing = True
    xlim = ax.get_xlim()
    ax_cumul.set_xlim(xlim)
    try:
        ax_total.set_xlim(xlim)
        fig3.canvas.draw_idle()
    except NameError:
        pass
    _update_titles(xlim)
    _update_ticks(xlim)
    fig2.canvas.draw_idle()
    fig.canvas.draw_idle()
    _xlim_syncing = False

def _sync_from_cumul(ax):
    global _xlim_syncing
    if _xlim_syncing:
        return
    _xlim_syncing = True
    xlim = ax.get_xlim()
    ax_status.set_xlim(xlim)  # sharex propagates to altaz and band
    try:
        ax_total.set_xlim(xlim)
        fig3.canvas.draw_idle()
    except NameError:
        pass
    _update_titles(xlim)
    _update_ticks(xlim)
    fig.canvas.draw_idle()
    fig2.canvas.draw_idle()
    _xlim_syncing = False

ax_status.callbacks.connect('xlim_changed', _sync_from_timeline)
ax_cumul.callbacks.connect('xlim_changed', _sync_from_cumul)

# Set initial ticks for the current view
_update_ticks((cxlim0, cxlim1))
fig.canvas.draw_idle()
fig2.canvas.draw_idle()
try:
    fig3.canvas.draw_idle()
except NameError:
    pass

# Cumulative state totals

In [ ]:
# Cumulative total hours in each state over the date range
# Independent of the timeline/per-night plots above

STACKED_TOTAL = stacked_total

TOTAL_LINESTYLE = {
    'OPERATIONAL': '-',
    'FAULT': '-',
    'WEATHER': '-',
    'DOWNTIME': '--',
    'UNKNOWN': '-',
    'IDLE': '-',
}

# Close previous figure if re-running
try:
    plt.close(fig3)
except NameError:
    pass

if SHOW_NIGHT_LENGTH:
    fig3, (ax_total, ax_nightlen) = plt.subplots(
        2, 1, figsize=(FIGSIZE_TOTAL[0], FIGSIZE_TOTAL[1] + 1),
        gridspec_kw={'height_ratios': [4, 1]}, sharex=True)
else:
    fig3, ax_total = plt.subplots(figsize=FIGSIZE_TOTAL)
    ax_nightlen = None

# Build continuous cumulative curves with intra-night detail.
# Each curve is a list of (datetime, cumul_hours) points.
_total_running = {st: 0.0 for st in TOTAL_STATES}
_total_curves = {st: [] for st in TOTAL_STATES}
_shutter_running = 0.0
_shutter_curve = []
_camera_running = 0.0
_camera_curve = []
_overhead_running = 0.0
_overhead_curve = []

# Build overhead (readout + slew) intervals from consecutive exposures
_sorted_cam_ivs = sorted(camera_shutter_intervals)
_overhead_ivs = []
for _oi, (_ot0, _ot1) in enumerate(_sorted_cam_ivs):
    if _oi + 1 < len(_sorted_cam_ivs):
        _ot0_next = _sorted_cam_ivs[_oi + 1][0]
        _ogap_s = (_ot0_next - _ot1).total_seconds()
        if 0 < _ogap_s <= MAX_SLEW_GAP_S:
            _overhead_ivs.append((_ot1, _ot0_next))
        elif _ogap_s > 0:
            _overhead_ivs.append((_ot1, _ot1 + pd.Timedelta(seconds=READOUT_S)))
    else:
        _overhead_ivs.append((_ot1, _ot1 + pd.Timedelta(seconds=READOUT_S)))

for sunset, sunrise in plot_twilight_spans:
    sunset_pd = pd.Timestamp(sunset)
    sunrise_pd = min(pd.Timestamp(sunrise), _data_now_naive)

    # Start of night: record current totals (flat from previous sunrise)
    for st in TOTAL_STATES:
        _total_curves[st].append((sunset_pd, _total_running[st]))
    _shutter_curve.append((sunset_pd, _shutter_running))
    _camera_curve.append((sunset_pd, _camera_running))

    # Collect and sort all state intervals clipped to this night
    night_events = []
    for start, end, state in intervals_naive:
        if state not in TOTAL_STATES:
            continue
        s = max(start, sunset_pd)
        e = min(end, sunrise_pd)
        if s < e:
            night_events.append((s, e, state))
    night_events = filter_all_during_downtime(night_events)
    night_events = filter_weather_during_operational(night_events)
    night_events = filter_unknown_with_other_states(night_events)
    night_events = filter_idle_during_weather(night_events)
    night_events = filter_weather_during_fault(night_events)
    night_events = filter_idle_during_fault(night_events)
    night_events.sort(key=lambda x: x[0])

    # Step through state intervals, adding points at each transition
    for s, e, state in night_events:
        dt_hours = (e - s).total_seconds() / 3600.0
        _total_curves[state].append((s, _total_running[state]))
        _total_running[state] += dt_hours
        _total_curves[state].append((e, _total_running[state]))

    # Shutter intervals clipped to this night
    sh_events = []
    for s_start, s_end in shutter_open_intervals:
        s = max(s_start, sunset_pd)
        e = min(s_end, sunrise_pd)
        if s < e:
            sh_events.append((s, e))
    sh_events.sort()

    for s, e in sh_events:
        dt_hours = (e - s).total_seconds() / 3600.0
        _shutter_curve.append((s, _shutter_running))
        _shutter_running += dt_hours
        _shutter_curve.append((e, _shutter_running))

    # Camera shutter intervals clipped to this night
    cam_events = []
    for c_start, c_end in camera_shutter_intervals:
        c_s = max(c_start, sunset_pd)
        c_e = min(c_end, sunrise_pd)
        if c_s < c_e:
            cam_events.append((c_s, c_e))
    cam_events.sort()
    for c_s, c_e in cam_events:
        dt_hours = (c_e - c_s).total_seconds() / 3600.0
        _camera_curve.append((c_s, _camera_running))
        _camera_running += dt_hours
        _camera_curve.append((c_e, _camera_running))

    # Overhead (readout + slew) intervals clipped to this night
    _ovh_events = []
    for _os, _oe in _overhead_ivs:
        _s = max(_os, sunset_pd)
        _e = min(_oe, sunrise_pd)
        if _s < _e:
            _ovh_events.append((_s, _e))
    _ovh_events.sort()
    _overhead_curve.append((sunset_pd, _overhead_running))
    for _s, _e in _ovh_events:
        _overhead_curve.append((_s, _overhead_running))
        _overhead_running += (_e - _s).total_seconds() / 3600.0
        _overhead_curve.append((_e, _overhead_running))

    # End of night: record totals at sunrise (clipped to now for in-progress night)
    for st in TOTAL_STATES:
        _total_curves[st].append((sunrise_pd, _total_running[st]))
    _shutter_curve.append((sunrise_pd, _shutter_running))
    _camera_curve.append((sunrise_pd, _camera_running))
    _overhead_curve.append((sunrise_pd, _overhead_running))

if STACKED_TOTAL:
    _all_xs_set = set()
    for state in TOTAL_STATES:
        for t, _ in _total_curves[state]:
            _all_xs_set.add(float(cx(mdates.date2num(t))))
    _all_xs_st = np.array(sorted(_all_xs_set))
    _interp_totals = {}
    for state in TOTAL_STATES:
        if _total_curves[state]:
            _xs_st = [float(cx(mdates.date2num(t))) for t, _ in _total_curves[state]]
            _ys_st = [v for _, v in _total_curves[state]]
            _interp_totals[state] = np.interp(_all_xs_st, _xs_st, _ys_st)
        else:
            _interp_totals[state] = np.zeros_like(_all_xs_st)
    _stack_bottom = np.zeros_like(_all_xs_st)
    for state in TOTAL_STATES:
        _stack_top = _stack_bottom + _interp_totals[state]
        if np.any(_interp_totals[state] > 0):
            ax_total.fill_between(_all_xs_st, _stack_bottom, _stack_top,
                                  color=COLORS[state], alpha=0.3,
                                  linewidth=0.5, edgecolor=COLORS[state])
        _stack_bottom = _stack_top
else:
    for state in TOTAL_STATES:
        if not _total_curves[state]:
            continue
        xs = [float(cx(mdates.date2num(t))) for t, _ in _total_curves[state]]
        ys = [v for _, v in _total_curves[state]]
        ax_total.plot(xs, ys, color=COLORS[state], linewidth=2,
                      linestyle=TOTAL_LINESTYLE[state],
                      label=state, alpha=0.6)

# Shutter open
if _shutter_curve:
    xs = [float(cx(mdates.date2num(t))) for t, _ in _shutter_curve]
    ys = [v for _, v in _shutter_curve]
    ax_total.plot(xs, ys, color='black', linewidth=2,
                  linestyle='-', label='Dome open', alpha=0.7)

# Camera shutter open + overhead fill
if _camera_curve and _overhead_curve:
    _xs_cam = [float(cx(mdates.date2num(t))) for t, _ in _camera_curve]
    _ys_cam = [v for _, v in _camera_curve]
    _xs_ovh = [float(cx(mdates.date2num(t))) for t, _ in _overhead_curve]
    _ys_ovh = [v for _, v in _overhead_curve]
    _xs_fill = sorted(set(_xs_cam) | set(_xs_ovh))
    _ys_fill_bot = np.interp(_xs_fill, _xs_cam, _ys_cam)
    _ys_fill_top = _ys_fill_bot + np.interp(_xs_fill, _xs_ovh, _ys_ovh)
    ax_total.fill_between(_xs_fill, _ys_fill_bot, _ys_fill_top,
                          alpha=0.15, color='black')
    ax_total.plot(_xs_cam, _ys_cam, color='black', linewidth=2,
                  linestyle='--', label='Camera exposing', alpha=0.9)
    ax_total.plot(_xs_fill, _ys_fill_top, color='#555555', linewidth=1.5,
                  linestyle=':', alpha=0.8, label='+ readout + slew')
elif _camera_curve:
    _xs_cam = [float(cx(mdates.date2num(t))) for t, _ in _camera_curve]
    _ys_cam = [v for _, v in _camera_curve]
    ax_total.plot(_xs_cam, _ys_cam, color='black', linewidth=2,
                  linestyle='--', label='Camera exposing', alpha=0.9)

ax_total.set_ylabel('Cumulative hours', fontsize=9)

# Right axis: fraction of cumulative available twilight-to-twilight time
_cumul_night_hours = []
_night_total = 0.0
for sunset, sunrise in plot_twilight_spans:
    _night_total += (pd.Timestamp(sunrise) - pd.Timestamp(sunset)).total_seconds() / 3600.0
    _cumul_night_hours.append(_night_total)

_state_total = sum(_total_running.values())
if _state_total > _night_total + 0.1:
    print(f"WARNING: state total {_state_total:.2f}h > available {_night_total:.2f}h"
          f" (excess {_state_total - _night_total:.2f}h)")
    for _st, _v in sorted(_total_running.items(), key=lambda x: -x[1]):
        if _v > 0:
            print(f"  {_st}: {_v:.2f}h")
    print()
    for _ss, _sr in plot_twilight_spans:
        _ss_pd = pd.Timestamp(_ss)
        _sr_pd = min(pd.Timestamp(_sr), _data_now_naive)
        _nav = (_sr_pd - _ss_pd).total_seconds() / 3600
        _night_ivs = [(s, e, st) for s, e, st in intervals_naive
                      if st in TOTAL_STATES and max(s, _ss_pd) < min(e, _sr_pd)]
        _night_ivs = [(max(s, _ss_pd), min(e, _sr_pd), st) for s, e, st in _night_ivs]
        _night_ivs = filter_all_during_downtime(_night_ivs)
        _night_ivs = filter_weather_during_operational(_night_ivs)
        _night_ivs = filter_unknown_with_other_states(_night_ivs)
        _night_ivs = filter_idle_during_weather(_night_ivs)
        _night_ivs = filter_weather_during_fault(_night_ivs)
        _night_ivs = filter_idle_during_fault(_night_ivs)
        _nst = sum((e - s).total_seconds() / 3600 for s, e, _ in _night_ivs)
        if _nst > _nav + 0.01:
            _novlp = [
                (st1, st2, (min(e1,e2)-max(s1,s2)).total_seconds()/3600)
                for i,(s1,e1,st1) in enumerate(_night_ivs)
                for j,(s2,e2,st2) in enumerate(_night_ivs)
                if j > i and st1 != st2 and s1 < e2 and s2 < e1
            ]
            _ovlp_str = ", ".join(f"{a}+{b}:{h:.3f}h" for a,b,h in _novlp) if _novlp else "no pairwise overlaps"
            print(f"  Night {_ss_pd.date()}: state={_nst:.3f}h avail={_nav:.3f}h"
                  f" excess={_nst-_nav:.3f}h  {_ovlp_str}")

if _night_total > 0:
    ax_frac = ax_total.twinx()
    ax_frac.set_ylabel('Fraction of available time', fontsize=9)
    # Sync the right axis limits to match the left axis scaled by total night hours
    def _sync_frac_ylim(ax):
        lo, hi = ax_total.get_ylim()
        ax_frac.set_ylim(lo / _night_total, hi / _night_total)
    _sync_frac_ylim(ax_total)
    ax_total.callbacks.connect('ylim_changed', _sync_frac_ylim)

if plot_twilight_spans:
    _d0 = pd.Timestamp(plot_twilight_spans[0][0]).strftime('%Y-%m-%d')
    _d1 = pd.Timestamp(plot_twilight_spans[-1][0]).strftime('%Y-%m-%d')
    _stacked_suffix = ' (stacked)' if STACKED_TOTAL else ''
    ax_total.set_title(f'Cumulative State Totals{_stacked_suffix} ({_d0} to {_d1} UTC)', fontsize=10)

if SHOW_NIGHT_LENGTH:
    _nl_xs, _nl_ys = [], []
    for sunset, sunrise in plot_twilight_spans:
        _ss_pd = pd.Timestamp(sunset)
        _sr_pd = pd.Timestamp(sunrise)
        _nh = (_sr_pd - _ss_pd).total_seconds() / 3600.0
        _x0 = float(cx(mdates.date2num(_ss_pd)))
        _x1 = float(cx(mdates.date2num(_sr_pd)))
        _nl_xs.append((_x0 + _x1) / 2)
        _nl_ys.append(_nh)
    ax_nightlen.plot(_nl_xs, _nl_ys, color='black', linewidth=1.5, alpha=0.7)
    ax_nightlen.set_ylabel('Night\nlength (h)', fontsize=8)
    ax_nightlen.locator_params(axis='y', nbins=3)
    ax_nightlen.grid(axis='y', alpha=0.3)

shade_daytime(ax_total)
if SHOW_NIGHT_LENGTH:
    shade_daytime(ax_nightlen)
    setup_xaxis(ax_total, show_labels=False)
    setup_xaxis(ax_nightlen, show_labels=True)
    plt.setp(ax_nightlen.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax_nightlen.set_xlabel('Date (UTC)', fontsize=9)
else:
    setup_xaxis(ax_total, show_labels=True)
    plt.setp(ax_total.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax_total.set_xlabel('Date (UTC)', fontsize=9)
ax_total.grid(axis='y', alpha=0.3)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
_pct = (lambda v: f' ({100*v/_night_total:.1f}%)') if _night_total > 0 else (lambda v: '')
_tail_legend = [
    Line2D([0], [0], color='black', linewidth=2, alpha=0.7, label='Dome open' + _pct(_shutter_running)),
    Line2D([0], [0], color='black', linewidth=2, linestyle='--', label='Camera exposing' + _pct(_camera_running)),
    Line2D([0], [0], color='#555555', linewidth=1.5, linestyle=':',
           label='+ readout + slew' + _pct(_camera_running + _overhead_running)),
]
if STACKED_TOTAL:
    total_legend = [
        Patch(facecolor=COLORS[st], alpha=0.3, label=st + _pct(_total_running[st])) for st in TOTAL_STATES
    ] + _tail_legend
else:
    total_legend = [
        Line2D([0], [0], color=COLORS[st], linewidth=2, alpha=0.6,
               linestyle=TOTAL_LINESTYLE[st], label=st + _pct(_total_running[st])) for st in TOTAL_STATES
    ] + _tail_legend
ax_total.legend(handles=total_legend, loc='upper left', fontsize=8)

fig3.subplots_adjust(bottom=0.15, hspace=0.05 if SHOW_NIGHT_LENGTH else 0)

def _sync_from_total(ax):
    global _xlim_syncing
    if _xlim_syncing:
        return
    _xlim_syncing = True
    xlim = ax.get_xlim()
    ax_status.set_xlim(xlim)
    ax_cumul.set_xlim(xlim)
    _update_titles(xlim)
    _update_ticks(xlim)
    fig.canvas.draw_idle()
    fig2.canvas.draw_idle()
    _xlim_syncing = False
ax_total.callbacks.connect('xlim_changed', _sync_from_total)
plt.show()

# Observatory Status Statistics

In [ ]:
# Per-night hours in each observatory state
_stats_states = ['OPERATIONAL', 'FAULT', 'WEATHER', 'DOWNTIME', 'UNKNOWN', 'IDLE']
_stats_cols = ['Night length'] + _stats_states + ['Dome open', 'Camera exposing']
_stats_nights = []  # list of dicts, one per night
_stats_dates = []   # night labels (sunset date)

for _sunset, _sunrise in plot_twilight_spans:
    _sunset_pd = pd.Timestamp(_sunset)
    _sunrise_pd = min(pd.Timestamp(_sunrise), _data_now_naive)
    _night_h = (_sunrise_pd - _sunset_pd).total_seconds() / 3600.0

    _nivs = []
    for _s, _e, _st in intervals_naive:
        if _st not in _stats_states:
            continue
        _cs = max(_s, _sunset_pd)
        _ce = min(_e, _sunrise_pd)
        if _cs < _ce:
            _nivs.append((_cs, _ce, _st))

    _nivs = filter_all_during_downtime(_nivs)
    _nivs = filter_weather_during_operational(_nivs)
    _nivs = filter_unknown_with_other_states(_nivs)
    _nivs = filter_idle_during_weather(_nivs)
    _nivs = filter_weather_during_fault(_nivs)
    _nivs = filter_idle_during_fault(_nivs)

    _cumul = {st: 0.0 for st in _stats_states}
    for _cs, _ce, _st in _nivs:
        _cumul[_st] += (_ce - _cs).total_seconds() / 3600.0
    _ofdu_nivs = [(cs, ce, st) for cs, ce, st in _nivs
                  if st in ('OPERATIONAL', 'FAULT', 'DOWNTIME', 'UNKNOWN', 'IDLE')]
    if _ofdu_nivs:
        _last_ofdu_end = max(_ce for _, _ce, _ in _ofdu_nivs)
        _end_gap_s = (_sunrise_pd - _last_ofdu_end).total_seconds()
        if 0 < _end_gap_s < 120:
            _last_st = max(_ofdu_nivs, key=lambda x: x[1])[2]
            _cumul[_last_st] += _end_gap_s / 3600.0

    _sh = 0.0
    for _ss, _se in shutter_open_intervals:
        _cs = max(_ss, _sunset_pd)
        _ce = min(_se, _sunrise_pd)
        if _cs < _ce:
            _sh += (_ce - _cs).total_seconds() / 3600.0

    _row = {'Night length': _night_h}
    _row.update(_cumul)
    _row['Dome open'] = _sh

    _cam = 0.0
    for _cs2, _ce2 in camera_shutter_intervals:
        _cs = max(_cs2, _sunset_pd)
        _ce = min(_ce2, _sunrise_pd)
        if _cs < _ce:
            _cam += (_ce - _cs).total_seconds() / 3600.0
    _row['Camera exposing'] = _cam
    _stats_nights.append(_row)
    _ofdu = sum(_cumul[st] for st in ('OPERATIONAL', 'FAULT', 'DOWNTIME', 'UNKNOWN', 'IDLE'))
    _date_lbl = _sunset_pd.strftime('%m-%d')
    if _night_h - _ofdu > 1 / 3600:
        _date_lbl += '*'
    _stats_dates.append(_date_lbl)


# Build DataFrame: nights as rows, states as columns
_df = pd.DataFrame(_stats_nights, index=_stats_dates, columns=_stats_cols)
_df.index.name = 'Night'
if SHOW_OFDU:
    _df.insert(1, 'O+F+D+U', _df[['OPERATIONAL','FAULT','DOWNTIME','UNKNOWN','IDLE']].sum(axis=1))

# Append summary rows
_summary = pd.DataFrame({
    c: [_df[c].sum(), _df[c].mean(), _df[c].median(), median_abs_deviation(_df[c].values, scale="normal")]
    for c in _df.columns
}, index=['Total', 'Mean', 'Median', 'σ_MAD'])
_df_full = pd.concat([_df, _summary])

_n = len(plot_twilight_spans)
_title = f"Hours per night ({_n} nights, {title_start} to {title_end} UTC)\nFAULT excludes time during DOWNTIME; WEATHER excludes time during DOWNTIME"
_table_str = _df_full.round(2).to_string()
_lines = _table_str.split('\n')
_width = max(len(l) for l in _lines)
_sep = '-' * _width
_lines.insert(1 + len(_df), _sep)
print(_title)
print(_sep)
print('\n'.join(_lines))